# SEED-VII EEGNet × LoRA-LLM ModelScope Pipeline

本 Notebook 使用 **ModelScope Dataset API** 拉取 `DEREKVERSE/SEED-VII` 数据集里的 `EEG_preprocessed/1-20.mat` 和你的 `text_protocol.csv`。

关键修正：
- EEG `.mat` 是 HDF5 MATLAB v7.3，变量名为 `1`...`80`，每个变量真实 MATLAB 尺寸为 `62×N`；读取器同时兼容 h5py 暴露为 `(62,N)` 或 `(N,62)`。
- LLM Tower 输入 **只使用参考库 CSV 协议中的 L2 文本**：80 clips 对应 80 条 `l2_text`。
- 对比学习正负样本只由聚合三分类标签决定：同类为正，异类为负，不看 subject/video。


In [ ]:
# 0. 环境准备
!python -V
!pip install -q -r requirements.txt
!pip install -q -e .


In [ ]:
# 1. 路径配置
from pathlib import Path
import os, yaml

PROJECT = Path.cwd()
WORK = Path('/mnt/workspace')
WORK.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'DEREKVERSE/SEED-VII'
LOCAL_DATASET_DIR = WORK / 'seedvii_ms_dataset'
NPZ_DIR = WORK / 'seedvii_npz'
RUN_DIR = WORK / 'seedvii_contrastive_runs' / 'run_valence3_l2text'
MODEL_DIR = WORK / 'models' / 'Qwen2.5-0.5B-Instruct'

print('PROJECT=', PROJECT)
print('LOCAL_DATASET_DIR=', LOCAL_DATASET_DIR)
print('NPZ_DIR=', NPZ_DIR)
print('RUN_DIR=', RUN_DIR)


## 2. 通过 ModelScope Dataset API 拉取 1-20.mat 和 text_protocol.csv

脚本会先 listing/search 数据集文件，筛选：

```text
1.mat ... 20.mat
text_protocol*.csv
videoid_to_emotion*.csv, if present
```

注意这里使用的是 dataset 协议：`repo_type='dataset'` / `dataset_snapshot_download`，不是模型下载协议。


In [ ]:
# 如果是私有数据集，请设置 token；公开数据集可不设。
import os
os.environ['MODELSCOPE_TOKEN'] = 'ms-2460c377-dcfc-4cb5-86d5-635cfa6ea9e2'

!python -m EEG_OPUS1.seedvii_modal_contrastive_lora.seedvii_contrastive.scripts.download_modelscope_seedvii \
  --dataset-id {DATASET_ID} \
  --local-dir {LOCAL_DATASET_DIR} \
  --max-workers 4


In [ ]:
# 3. 自动发现下载后的 EEG_ROOT 和 TEXT_CSV
from EEG_OPUS1.seedvii_modal_contrastive_lora.seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths
EEG_ROOT, TEXT_CSV = find_downloaded_paths(LOCAL_DATASET_DIR)
print('EEG_ROOT=', EEG_ROOT)
print('TEXT_CSV=', TEXT_CSV)
assert EEG_ROOT is not None, '没有找到 1-20.mat 所在目录'
assert TEXT_CSV is not None, '没有找到 text_protocol*.csv；LLM Tower 必须使用你的 L2 文本协议'



## 4. 下载/指定 LLM

推荐小模型先跑通，例如 `Qwen/Qwen2.5-0.5B-Instruct`。如果你已在 ModelScope 实例中挂载本地模型目录，可以直接修改 `MODEL_DIR`。


In [ ]:
from modelscope import snapshot_download

if not MODEL_DIR.exists():
    model_path = snapshot_download('Qwen/Qwen2.5-0.5B-Instruct', cache_dir=str(WORK / 'models'))
    print('downloaded:', model_path)
    MODEL_DIR = Path(model_path)
else:
    print('use existing model:', MODEL_DIR)


## 5. NPZ 预处理

每个 subject/trial 独立处理：中间 60%，4s non-overlap windows，最多每 clip 采样固定窗口数。


In [ ]:
if not (NPZ_DIR / 'index.csv').exists():
    !python -m EEG_OPUS1.seedvii_modal_contrastive_lora.seedvii_contrastive.scripts.preprocess_npz \
      --input-root {EEG_ROOT} \
      --output-dir {NPZ_DIR} \
      --subjects 1-20 \
      --window-sec 4 --stride-sec 4 \
      --center-ratio 0.60 \
      --max-windows-per-clip 12 \
      --shard-size 512
else:
    print('NPZ index already exists:', NPZ_DIR / 'index.csv')



In [ ]:
# 6. 写入本次运行配置：注意 text_csv_path 指向你的 L2 text_protocol.csv
base_cfg_path = PROJECT / 'configs' / 'modelscope_default.yaml'
cfg = yaml.safe_load(open(base_cfg_path, 'r', encoding='utf-8'))
cfg['data']['modelscope_dataset_id'] = DATASET_ID
cfg['data']['local_dataset_dir'] = str(LOCAL_DATASET_DIR)
cfg['data']['eeg_root'] = str(EEG_ROOT)
cfg['data']['text_csv_path'] = str(TEXT_CSV)
cfg['data']['npz_dir'] = str(NPZ_DIR)
cfg['runtime']['output_dir'] = str(RUN_DIR)
cfg['model']['llm']['model_name_or_path'] = str(MODEL_DIR)

# 显存不够可调小 batch 或开启 gradient checkpointing
# cfg['train']['batch_size'] = 24
# cfg['model']['llm']['gradient_checkpointing'] = True

run_cfg = RUN_DIR / 'config.yaml'
RUN_DIR.mkdir(parents=True, exist_ok=True)
yaml.safe_dump(cfg, open(run_cfg, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print(open(run_cfg, 'r', encoding='utf-8').read())


## 7. 训练 / 继续训练

`train.resume=true` 时会自动读取 `last.pt` 继续训练；验证集 macro-F1 最优保存 `best.pt`。


In [ ]:
!python -m EEG_OPUS1.seedvii_modal_contrastive_lora.seedvii_contrastive.scripts.train_contrastive --config {run_cfg}


## 8. EEG 编码 / 推理

分类时使用 80 条 L2 文本按三分类聚合得到 LLM 文本原型，再与 EEG embedding 做相似度。


In [ ]:
BEST = RUN_DIR / 'best.pt'
OUT_EMB = RUN_DIR / 'val_embeddings.npz'
!python -m EEG_OPUS1.seedvii_modal_contrastive_lora.seedvii_contrastive.scripts.encode_eeg \
  --config {run_cfg} \
  --checkpoint {BEST} \
  --split val \
  --out {OUT_EMB}
print('saved:', OUT_EMB)
